# Helix CPG Market Gap Analysis
**Lead AI Engineer:** Ryan Nii Akwei Brown  
**Date:** April 2026

For a strategic consultancy like Helix CPG Partners, committing to a new product line without empirical evidence is a significant risk. This analysis replaces guesswork with data-backed insights. By processing a representative sample of global food data, we locate the 'Blue Ocean' where consumer demand for high-protein, low-sugar snacks is currently unmet. The goal is to provide a clear, defensive justification for a specific R&D investment before any capital is committed.

For a detailed explanation of the dataset used in this analysis, please refer to the [Open Food Facts Data Fields Guide](https://static.openfoodfacts.org/data/data-fields.txt).

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import re
import os
import matplotlib.pyplot as plt

---
## Step 1: Data Ingestion (RAM-Safe Approach)
To handle the massive 12GB Open Food Facts dataset, we use a memory-efficient chunked ingestion strategy. By processing the data in chunks of 50,000 rows and immediately downcasting numerical columns to float32, we ensure the analysis remains stable on standard hardware.

In [2]:
DATA_URL = "https://world.openfoodfacts.org/data/en.openfoodfacts.org.products.csv"
STREAM_LIMIT = 500000
SAMPLE_SIZE = 200000
CHUNK_SIZE = 50000
OUTPUT_FILE = "cleaned_snack_data.csv"

def ingest_data(url, stream_n, sample_n):
    cols = [
        'product_name',
        'categories_tags',
        'sugars_100g',
        'proteins_100g',
        'ingredients_text'
    ]

    print(f"Ingesting data in chunks of {CHUNK_SIZE}...")
    chunks = []
    rows_collected = 0
    
    # RAM-Safe Ingestion with chunksize and float32 conversion
    for chunk in pd.read_csv(url, sep='\t', chunksize=CHUNK_SIZE, usecols=cols, on_bad_lines='skip', low_memory=False, engine='c'):
        # Immediate conversion to save memory
        chunk['sugars_100g'] = pd.to_numeric(chunk['sugars_100g'], errors='coerce').astype('float32')
        chunk['proteins_100g'] = pd.to_numeric(chunk['proteins_100g'], errors='coerce').astype('float32')
        
        chunks.append(chunk)
        rows_collected += len(chunk)
        if rows_collected >= stream_n:
            break
            
    df_pool = pd.concat(chunks, ignore_index=True)
    print(f"Shuffling and selecting a random sample of {sample_n} rows...")
    df = df_pool.sample(n=min(sample_n, len(df_pool)), random_state=7)

    return df

---
## Step 2: Data Cleaning & Biological Boundaries
Crowd-sourced data often contains errors. We enforce biological boundaries (0-100g) for nutritional values to ensure integrity. We also track data removal to generate a quality report.

In [3]:
def clean_data(df):
    print("Cleaning data and applying boundaries...")

    # Store initial metrics for the quality report
    stats = {"initial_rows": len(df)}

    # Let's remove rows missing critical information
    df = df.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])
    stats["after_null_removal"] = len(df)

    # Let's remove impossible nutritional values
    df = df[
        (df['sugars_100g'] >= 0) & (df['sugars_100g'] <= 100) &
        (df['proteins_100g'] >= 0) & (df['proteins_100g'] <= 100)
    ]
    stats["final_clean_rows"] = len(df)

    # Let's print the Data Quality Report
    print("\n--- DATA QUALITY REPORT ---")
    print(f"Total rows sampled: {stats['initial_rows']}")
    print(f"Rows dropped due to missing values: {stats['initial_rows'] - stats['after_null_removal']}")
    print(f"Rows dropped due to biological boundary violations: {stats['after_null_removal'] - stats['final_clean_rows']}")
    print(f"Total usable records: {stats['final_clean_rows']}")
    print("---------------------------\n")

    return df

## Step 3: Market Categorization (User Story 2)
The raw dataset contains thousands of granular tags. To make the analysis
actionable for Helix CPG Partners, let's map these into five high-level
business buckets. This allows us to ignore noise and focus specifically
on the competitive snack landscape.

In [4]:
def categorize_market(df):
    print("Categorizing products into business buckets...")
    category_map = {
        'snack': 'Snacks',
        'biscuit': 'Snacks',
        'confectionery': 'Snacks',
        'beverage': 'Beverages',
        'dairy': 'Dairy',
        'plant-based': 'Plant-based',
        'cereal': 'Cereals'
    }
    df['primary_category'] = 'Other'
    df['categories_tags'] = df['categories_tags'].fillna('').astype(str)
    for keyword, label in category_map.items():
        mask = df['categories_tags'].str.contains(keyword, case=False, na=False)
        df.loc[mask, 'primary_category'] = label
    return df[df['primary_category'] != 'Other'].copy()

---
## Step 4: Analytical Engine & Ingredient Mining
We calculate a Nutrient Density Score to identify efficiency (Protein per unit of Sugar). Additionally, we use Regular Expressions to mine ingredients for specific protein sources and sodium markers, helping to define the product differentiator.

In [5]:
def calculate_metrics(df):
    print("Calculating Nutrient Density and mining ingredients...")
    df['nutrient_density_score'] = df['proteins_100g'] / (df['sugars_100g'] + 1)
    
    target_cat = 'Plant-based'
    p_median = df['proteins_100g'].median()
    cluster = df[(df['primary_category'] == target_cat) & (df['proteins_100g'] > p_median)]
    
    # Regex to identify protein sources
    protein_regex = re.compile(r'\b(soy|pea|whey|nut|almond|peanut|lentil|bean|chickpea|egg|milk|casein|seeds|oats)\b', re.IGNORECASE)
    
    protein_sources = []
    high_sodium_flag = False
    
    for text in cluster['ingredients_text'].dropna():
        sources = protein_regex.findall(text.lower())
        protein_sources.extend(sources)
        
        # Flagging high sodium (salt) as a potential differentiator
        if re.search(r'\b(salt|sodium|chloride)\b', text.lower()):
            high_sodium_flag = True
            
    top_3_sources = [source for source, count in Counter(protein_sources).most_common(3)]
    
    if high_sodium_flag:
        print("Market Differentiator Alert: High sodium/salt presence detected in high-protein samples.")
        
    return df, top_3_sources

---
# Step 5: Strategic Market Mapping and Visual Validation
Before exporting to the dashboard, let's generate a strategic scatter plot
to confirm the distribution and verify that the Blue Ocean quadrant is statistically
visible. Note: Colors are matched to the Power BI theme for cross-platform consistency.

In [6]:
def visualize_summary(df):
    # Defining color hex codes to match Power BI theme
    color_palette = {
        'Cereals': '#001C7F',      # Dark Navy
        'Plant-based': '#68007E',  # Deep Purple
        'Snacks': '#D1398E',       # Pink
        'Dairy': '#E66C37',        # Orange
        'Beverages': '#0090FF'     # Bright Blue
    }

    plt.figure(figsize=(12, 7))
    for cat in df['primary_category'].unique():
        subset = df[df['primary_category'] == cat]
        plt.scatter(
            subset['sugars_100g'],
            subset['proteins_100g'],
            label=cat,
            alpha=0.6,
            s=25,
            color=color_palette.get(cat, '#808080')
        )

    # Adding executive benchmarks (Market Medians)
    plt.axvline(df['sugars_100g'].median(), color='#FF4B4B', linestyle='--', label='Market Sugar Median', linewidth=1.5)
    plt.axhline(df['proteins_100g'].median(), color='#2ECC71', linestyle='--', label='Market Protein Median', linewidth=1.5)

    plt.title('CPG Strategy Mapping: Identifying the High-Protein Blue Ocean Market Gap', fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Sugar Concentration (g/100g)', fontsize=11)
    plt.ylabel('Protein Concentration (g/100g)', fontsize=11)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, shadow=True)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

def main():
    df = ingest_data(DATA_URL, STREAM_LIMIT, SAMPLE_SIZE)
    df = clean_data(df)
    df = categorize_market(df)
    df, top_protein_sources = calculate_metrics(df)

    print("\n--- ANALYSIS COMPLETE ---")
    print(f"Top 3 Protein Sources identified via Regex: {top_protein_sources}")

    # Fixed Thresholds for Strategic Recommendation
    protein_threshold = 4.2
    sugar_threshold = 4.6
    print(f"Final Recommendation Thresholds: Protein > {protein_threshold}g, Sugar < {sugar_threshold}g")

    # Visual check
    visualize_summary(df)

    df.to_csv(OUTPUT_FILE, index=False)
    print(f"File exported as {OUTPUT_FILE}")

if __name__ == "__main__":
    main()